# DeepDTA on KIBA (CNN, as in the base paper)

Run the cells **in order, top to bottom**. Before you start: **Runtime > Change runtime type > GPU**.

What this notebook does:
1. Loads the KIBA data from the DeepDTA GitHub repo and uses the paper's train and test folds.
2. Trains the paper's CNN model for up to 100 epochs with a validation fold and early stopping (SMILES length 100, protein length 1000).
3. Measures CI, MSE, r2m and AUPR on the test fold.
4. Saves test results for the accuracy graph and 5 example pairs for the website.
5. Converts the trained model to ONNX, checks it against the original, and downloads one ZIP file.

Training is the long part (roughly 1.5 to 3 hours on a free GPU). Keep this browser tab open.

## 1. Setup

In [ ]:
import os, json, pickle, shutil, glob
from collections import OrderedDict

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, average_precision_score

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU found:", gpus if gpus else "NO GPU. Go to Runtime > Change runtime type > GPU, then run this cell again.")

## 2. Settings (KIBA values from the paper)

In [ ]:
MAX_SMILES_LENGTH = 100       # Section 2.3: KIBA
MAX_PROTEIN_LENGTH = 1000     # Section 2.3: KIBA
EMBEDDING_DIM = 128           # Section 2.3
NUM_FILTERS = 32              # 32, 64, 96 filters (Section 3.3)
SMILES_KERNEL = 4             # inside the paper's range (Table 2)
PROTEIN_KERNEL = 8            # inside the paper's range (Table 2)
DENSE_UNITS = (1024, 1024, 512)
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
BATCH_SIZE = 256
EPOCHS = 100
KIBA_THRESHOLD = 12.1         # Section 3.4: KIBA score >= 12.1 counts as binding (used for AUPR)
PATIENCE = 15                 # stop if validation loss has not improved for this many epochs
LR_PATIENCE = 5               # halve the learning rate if validation loss stalls for this many epochs
SEED = 42
tf.keras.utils.set_random_seed(SEED)   # same results every run
print("Settings ready")

## 3. Get the data from GitHub

In [ ]:
ZIP = "/content/deepdta.zip"
!wget -q https://github.com/hkmztrk/DeepDTA/archive/refs/heads/master.zip -O /content/deepdta.zip

if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1_000_000:
    from google.colab import files
    print("Download from GitHub failed. Upload DeepDTA-master.zip from your computer:")
    up = files.upload()
    ZIP = "/content/" + list(up.keys())[0]

!unzip -q -o "{ZIP}" -d /content/deepdta_repo
KIBA_DIR = glob.glob("/content/deepdta_repo/**/data/kiba", recursive=True)[0]
print("Using:", KIBA_DIR)
print(os.listdir(KIBA_DIR))

## 4. Load drugs, proteins and scores

In [ ]:
with open(os.path.join(KIBA_DIR, "ligands_can.txt")) as f:
    ligands = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(KIBA_DIR, "proteins.txt")) as f:
    proteins = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(KIBA_DIR, "Y"), "rb") as f:
    Y = pickle.load(f, encoding="latin1")

drug_ids = list(ligands.keys())
protein_ids = list(proteins.keys())
drugs = list(ligands.values())       # SMILES strings
targets = list(proteins.values())    # amino-acid sequences
affinity = np.array(Y, dtype=np.float64)   # KIBA scores. NaN = never measured. No transformation for KIBA.

print("Drugs:", len(drugs), "| Proteins:", len(targets))
print("Y shape:", affinity.shape)
assert affinity.shape == (len(drugs), len(targets)), "Y does not match the drug and protein lists"

drug_idx, target_idx = np.where(~np.isnan(affinity))
print("Known drug-target pairs:", len(drug_idx), "(paper: 118,254)")
print("KIBA score range:", round(np.nanmin(affinity), 3), "to", round(np.nanmax(affinity), 3))
assert len(drug_idx) == 118254, "Pair count differs from the paper's Table 1"

## 5. Paper's train and test folds, and the number encoding

In [ ]:
with open(os.path.join(KIBA_DIR, "folds", "train_fold_setting1.txt")) as f:
    train_folds = json.load(f)
with open(os.path.join(KIBA_DIR, "folds", "test_fold_setting1.txt")) as f:
    test_fold = json.load(f)

train_positions = np.array([p for fold in train_folds for p in fold])
test_positions = np.array(test_fold)

train_drugs, train_targets = drug_idx[train_positions], target_idx[train_positions]
test_drugs, test_targets = drug_idx[test_positions], target_idx[test_positions]
y_train = affinity[train_drugs, train_targets]
y_test = affinity[test_drugs, test_targets]
print("Training pairs:", len(y_train), "| Test pairs:", len(y_test))
assert len(set(train_positions) & set(test_positions)) == 0, "Train and test overlap"

smiles_chars = sorted(set("".join(drugs)))
protein_chars = sorted(set("".join(targets)))
smiles_vocab = {ch: i + 1 for i, ch in enumerate(smiles_chars)}
protein_vocab = {ch: i + 1 for i, ch in enumerate(protein_chars)}

print("SMILES symbols:", len(smiles_vocab), "->", "".join(smiles_chars))
print("Contains lowercase aromatic letters (c, n, o, s):", [c for c in "cnos" if c in smiles_vocab])
print("Contains '[', '@', '/' or backslash:", [c for c in "[@/\\" if c in smiles_vocab])
print("Protein symbols:", len(protein_vocab), "->", "".join(protein_chars))

smiles_style = {
    "lowercase_aromatic": any(c in smiles_vocab for c in "cnosp"),
    "brackets": "[" in smiles_vocab,
    "stereo_chars": any(c in smiles_vocab for c in "@/\\"),
    "plus_or_minus": any(c in smiles_vocab for c in "+-"),
}
print("SMILES style found in this dataset:", smiles_style)

def encode(sequence, vocab, max_len):
    encoded = np.zeros(max_len, dtype=np.int32)   # zeros = padding
    for i, ch in enumerate(sequence[:max_len]):
        encoded[i] = vocab.get(ch, 0)
    return encoded

encoded_drugs = np.array([encode(s, smiles_vocab, MAX_SMILES_LENGTH) for s in drugs])
encoded_targets = np.array([encode(s, protein_vocab, MAX_PROTEIN_LENGTH) for s in targets])

X_drug_train, X_target_train = encoded_drugs[train_drugs], encoded_targets[train_targets]
X_drug_test, X_target_test = encoded_drugs[test_drugs], encoded_targets[test_targets]
print("Input shapes:", X_drug_train.shape, X_target_train.shape)

# ---- NEW: validation set -------------------------------------------------
# The paper's training data is split into 5 folds. We train on 4 of them and keep the
# 5th as a VALIDATION set, so we can watch for overfitting while training.
# The test fold is still never touched during training.
val_positions = np.array(train_folds[-1])
fit_positions = np.array([p for fold in train_folds[:-1] for p in fold])
assert len(set(fit_positions) & set(val_positions)) == 0, "fit and validation overlap"
assert len((set(fit_positions) | set(val_positions)) & set(test_positions)) == 0, "test leaked into training"

fit_drugs, fit_targets = drug_idx[fit_positions], target_idx[fit_positions]
val_drugs, val_targets = drug_idx[val_positions], target_idx[val_positions]
y_fit = affinity[fit_drugs, fit_targets]
y_val = affinity[val_drugs, val_targets]
X_drug_fit, X_target_fit = encoded_drugs[fit_drugs], encoded_targets[fit_targets]
X_drug_val, X_target_val = encoded_drugs[val_drugs], encoded_targets[val_targets]
print("Fit pairs:", len(y_fit), "| Validation pairs:", len(y_val), "| Test pairs:", len(y_test))

## 6. Optional: keep checkpoints in Google Drive
If Drive connects, training progress survives a disconnect. **If it shows an error, ignore it and carry on**: checkpoints then stay in Colab's temporary storage.

In [ ]:
WORK_DIR = "/content/kiba_work"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/DeepDTA_project/kiba_work"
    print("Drive connected.")
except Exception as e:
    print("Drive not connected, using temporary storage instead:", type(e).__name__)
os.makedirs(WORK_DIR, exist_ok=True)
print("Work folder:", WORK_DIR)

## 7. Build the model (two CNN blocks, as in the paper)

In [ ]:
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model

def cnn_branch(name, max_len, vocab_size, kernel_size):
    inp = Input(shape=(max_len,), name=name + "_input")
    x = Embedding(input_dim=vocab_size + 1, output_dim=EMBEDDING_DIM, name=name + "_embedding")(inp)
    for i, mult in enumerate((1, 2, 3), start=1):
        x = Conv1D(NUM_FILTERS * mult, kernel_size, activation="relu", name=f"{name}_conv{i}")(x)
    x = GlobalMaxPooling1D(name=name + "_pool")(x)
    return inp, x

drug_input, drug_features = cnn_branch("drug", MAX_SMILES_LENGTH, len(smiles_vocab), SMILES_KERNEL)
protein_input, protein_features = cnn_branch("protein", MAX_PROTEIN_LENGTH, len(protein_vocab), PROTEIN_KERNEL)

x = Concatenate(name="merge")([drug_features, protein_features])
x = Dense(DENSE_UNITS[0], activation="relu", name="fc1")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[1], activation="relu", name="fc2")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[2], activation="relu", name="fc3")(x)
output = Dense(1, activation="linear", name="score")(x)

model = Model(inputs=[drug_input, protein_input], outputs=output, name="DeepDTA_KIBA")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse")
model.summary()

## 8. Train (with a validation set, early stopping and best-epoch saving)
Each epoch the model is scored on the **validation** fold. Training stops by itself when the validation loss stops improving, and the **best** epoch's weights are kept (not the last epoch's).
**If the session stops partway** and Colab is still connected, run cells 1 to 7 again and then this cell: it resumes from the last saved epoch.

In [ ]:
CKPT_LAST = os.path.join(WORK_DIR, "kiba_last.keras")
CKPT_BEST = os.path.join(WORK_DIR, "kiba_best.keras")
LOG = os.path.join(WORK_DIR, "kiba_training_log.csv")
DONE = os.path.join(WORK_DIR, "training_done.flag")

start_epoch = 0
if os.path.exists(CKPT_LAST) and os.path.exists(LOG) and not os.path.exists(DONE):
    start_epoch = len(pd.read_csv(LOG))
    model = tf.keras.models.load_model(CKPT_LAST)
    print("Resuming from epoch", start_epoch)

if os.path.exists(DONE):
    model = tf.keras.models.load_model(CKPT_BEST)
    print("Training already finished. Loaded the best-epoch model.")
else:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(CKPT_LAST, save_best_only=False),
        tf.keras.callbacks.ModelCheckpoint(CKPT_BEST, monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=LR_PATIENCE, min_lr=1e-5, verbose=1),
        tf.keras.callbacks.CSVLogger(LOG, append=True),
    ]
    history = model.fit(
        [X_drug_fit, X_target_fit], y_fit,
        validation_data=([X_drug_val, X_target_val], y_val),
        epochs=EPOCHS, initial_epoch=start_epoch,
        batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2,
    )
    open(DONE, "w").write("done")
    print("Training finished.")

## 8b. Check for overfitting / underfitting
Reads the training log, plots training vs validation loss, and compares **train, validation and test MSE** all measured the same way (dropout off).

In [ ]:
import matplotlib.pyplot as plt

log = pd.read_csv(LOG)
best_epoch = int(log["val_loss"].idxmin())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(log["loss"], label="train loss (dropout on)")
ax[0].plot(log["val_loss"], label="validation loss")
ax[0].axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch = {best_epoch + 1}")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("MSE"); ax[0].set_title("Loss curves"); ax[0].legend()
ax[1].plot(log["loss"], label="train"); ax[1].plot(log["val_loss"], label="validation")
ax[1].set_yscale("log"); ax[1].set_xlabel("Epoch"); ax[1].set_title("Same, log scale"); ax[1].legend()
plt.tight_layout(); plt.show()

# Fair comparison: all three measured in inference mode (dropout off)
rng = np.random.default_rng(0)
sub = rng.choice(len(y_fit), size=min(20000, len(y_fit)), replace=False)
def mse_of(Xd, Xt, y):
    return mean_squared_error(y, model.predict([Xd, Xt], batch_size=512, verbose=0).ravel())
train_mse = mse_of(X_drug_fit[sub], X_target_fit[sub], y_fit[sub])
val_mse = mse_of(X_drug_val, X_target_val, y_val)
test_mse = mse_of(X_drug_test, X_target_test, y_test)
baseline_mse = float(np.var(y_val))   # error of always predicting the mean

print(f"Baseline (predict the mean) : {baseline_mse:.4f}")
print(f"Train MSE      : {train_mse:.4f}")
print(f"Validation MSE : {val_mse:.4f}")
print(f"Test MSE       : {test_mse:.4f}")
print(f"Val / Train ratio: {val_mse / train_mse:.1f}x")

if val_mse > 0.9 * baseline_mse:
    print("-> UNDERFITTING or a bug: barely better than predicting the mean.")
elif val_mse / train_mse > 3:
    print("-> Clear OVERFITTING: raise DROPOUT_RATE (e.g. 0.2-0.3), add L2, or reduce dense layer size.")
elif val_mse / train_mse > 1.5:
    print("-> Mild overfitting (normal for this model). Early stopping is already limiting it.")
else:
    print("-> Good fit: train and validation errors are close.")
if abs(val_mse - test_mse) > 0.03:
    print("-> Validation and test differ a lot: check the split.")

## 9. Test results (the test fold was never used for training or for choosing the best epoch)

In [ ]:
def concordance_index(y_true, y_pred):
    # Paper Eq. (7)-(8): share of pairs ranked in the right order; ties count 0.5
    total, pairs = 0.0, 0
    for i in range(len(y_true)):
        larger = y_true[i] > y_true
        diff = y_pred[i] - y_pred[larger]
        total += np.sum(diff > 0) + 0.5 * np.sum(diff == 0)
        pairs += np.sum(larger)
    return total / pairs

def r2m(y_obs, y_pred):
    # r2m as written in the DeepDTA repo's emetrics.py (observed values regressed on predictions)
    y_obs, y_pred = np.asarray(y_obs), np.asarray(y_pred)
    r2 = (np.sum((y_pred - y_pred.mean()) * (y_obs - y_obs.mean())) ** 2) / (
        np.sum((y_obs - y_obs.mean()) ** 2) * np.sum((y_pred - y_pred.mean()) ** 2))
    k = np.sum(y_obs * y_pred) / np.sum(y_pred * y_pred)
    r02 = 1 - np.sum((y_obs - k * y_pred) ** 2) / np.sum((y_obs - y_obs.mean()) ** 2)
    return r2 * (1 - np.sqrt(abs(r2 * r2 - r02 * r02)))

y_pred = model.predict([X_drug_test, X_target_test], batch_size=512, verbose=0).ravel()

ci = concordance_index(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2m_value = r2m(y_test, y_pred)
aupr = average_precision_score((y_test >= KIBA_THRESHOLD).astype(int), y_pred)

print("========== KIBA TEST RESULTS ==========")
print("CI  :", round(ci, 4),  " (paper: 0.863)")
print("MSE :", round(mse, 4), " (paper: 0.194)")
print("r2m :", round(r2m_value, 4), " (paper: 0.673)")
print("AUPR:", round(aupr, 4), " (paper: 0.788)")

## 10. Save the graph data and 5 example pairs

In [ ]:
OUT = os.path.join(WORK_DIR, "package", "kiba")
os.makedirs(OUT, exist_ok=True)

# metrics
metrics = {"dataset": "KIBA", "epochs": EPOCHS, "n_test": int(len(y_test)),
           "CI": round(float(ci), 4), "MSE": round(float(mse), 4),
           "r2m": round(float(r2m_value), 4), "AUPR": round(float(aupr), 4),
           "score_percentiles": {str(p): round(float(np.percentile(y_test, p)), 3) for p in (5, 25, 50, 75, 95)},
           "share_at_or_above_cutoff": round(float((y_test >= KIBA_THRESHOLD).mean()), 3)}
json.dump(metrics, open(os.path.join(OUT, "metrics.json"), "w"))

# 3,000 randomly chosen test pairs for the website graph: [predicted, measured]
rng = np.random.default_rng(0)
pick = rng.choice(len(y_test), size=min(3000, len(y_test)), replace=False)
points = [[round(float(y_pred[i]), 3), round(float(y_test[i]), 3)] for i in pick]
json.dump(points, open(os.path.join(OUT, "test_points.json"), "w"))

# 5 example pairs from the TEST fold, picked by measured value (not by how well the model did)
order = np.argsort(y_test)
examples = []
for q in (0.10, 0.30, 0.50, 0.70, 0.90):
    i = int(order[int(q * (len(order) - 1))])
    d, t = int(test_drugs[i]), int(test_targets[i])
    examples.append({
        "drug_id": drug_ids[d], "protein_id": protein_ids[t],
        "smiles": drugs[d], "sequence": targets[t],
        "measured": round(float(y_test[i]), 3),
    })
json.dump(examples, open(os.path.join(OUT, "examples.json"), "w"))

print(metrics)
for e in examples:
    print(e["drug_id"], "x", e["protein_id"], "| measured:", e["measured"], "| SMILES length:", len(e["smiles"]), "| protein length:", len(e["sequence"]))

## 11. Convert to ONNX and check it against the trained model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.save('/content/drive/MyDrive/deepdta_kiba_backup.keras')

In [ ]:
!pip install onnx onnxruntime

In [ ]:
import onnx, onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper

model.save(os.path.join(OUT, "deepdta_kiba.keras"))

config = {
    "dataset": "KIBA", "unit": "KIBA score", "binds_cutoff": KIBA_THRESHOLD,
    "max_smiles_length": MAX_SMILES_LENGTH, "max_protein_length": MAX_PROTEIN_LENGTH,
    "smiles_vocab": smiles_vocab, "protein_vocab": protein_vocab,
    "smiles_style": smiles_style,
}
json.dump(config, open(os.path.join(OUT, "config.json"), "w"))

nodes, inits = [], []
def init(name, arr):
    inits.append(numpy_helper.from_array(np.asarray(arr, dtype=np.float32), name))
    return name

def branch(prefix):
    in_name = prefix + "_input"
    emb_w = model.get_layer(prefix + "_embedding").get_weights()[0]
    nodes.append(helper.make_node("Gather", [init(prefix + "_emb", emb_w), in_name], [prefix + "_e"], axis=0))
    nodes.append(helper.make_node("Transpose", [prefix + "_e"], [prefix + "_t"], perm=[0, 2, 1]))
    x = prefix + "_t"
    for i in (1, 2, 3):
        w, b = model.get_layer(f"{prefix}_conv{i}").get_weights()
        w = np.transpose(w, (2, 1, 0))   # Keras (length, in, out) -> ONNX (out, in, length)
        nodes.append(helper.make_node("Conv", [x, init(f"{prefix}_cw{i}", w), init(f"{prefix}_cb{i}", b)], [f"{prefix}_c{i}"]))
        nodes.append(helper.make_node("Relu", [f"{prefix}_c{i}"], [f"{prefix}_r{i}"]))
        x = f"{prefix}_r{i}"
    nodes.append(helper.make_node("GlobalMaxPool", [x], [prefix + "_gp"]))
    nodes.append(helper.make_node("Flatten", [prefix + "_gp"], [prefix + "_f"], axis=1))
    return prefix + "_f"

d = branch("drug")
p = branch("protein")
nodes.append(helper.make_node("Concat", [d, p], ["cat"], axis=1))
x = "cat"
for ln in ("fc1", "fc2", "fc3", "score"):
    w, b = model.get_layer(ln).get_weights()
    out = "prediction" if ln == "score" else ln + "_o"
    nodes.append(helper.make_node("Gemm", [x, init(ln + "_w", w), init(ln + "_b", b)], [out]))
    if ln != "score":
        nodes.append(helper.make_node("Relu", [out], [ln + "_r"]))
        x = ln + "_r"

graph = helper.make_graph(
    nodes, "deepdta_kiba",
    [helper.make_tensor_value_info("drug_input", TensorProto.INT64, [None, MAX_SMILES_LENGTH]),
     helper.make_tensor_value_info("protein_input", TensorProto.INT64, [None, MAX_PROTEIN_LENGTH])],
    [helper.make_tensor_value_info("prediction", TensorProto.FLOAT, [None, 1])],
    initializer=inits,
)
onnx_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
onnx_model.ir_version = 8
onnx.checker.check_model(onnx_model)
ONNX_FILE = os.path.join(OUT, "model.onnx")
onnx.save(onnx_model, ONNX_FILE)
print("ONNX file size:", round(os.path.getsize(ONNX_FILE) / 1e6, 2), "MB")

# Compare on 500 REAL test pairs
n = 500
keras_out = model.predict([X_drug_test[:n], X_target_test[:n]], verbose=0).ravel()
sess = ort.InferenceSession(ONNX_FILE, providers=["CPUExecutionProvider"])
onnx_out = sess.run(None, {"drug_input": X_drug_test[:n].astype(np.int64),
                           "protein_input": X_target_test[:n].astype(np.int64)})[0].ravel()
diff = float(np.max(np.abs(keras_out - onnx_out)))
print("Largest difference between Keras and ONNX:", diff)
print("Keras:", keras_out[:3], "| ONNX:", onnx_out[:3])
assert diff < 1e-4, "ONNX does not match Keras. Do NOT use this file. Send the output to Claude."
print("PASSED: ONNX gives the same answers as Keras.")

## 12. Download everything as one ZIP

In [ ]:
from google.colab import files
zip_path = shutil.make_archive("/content/kiba_package", "zip", os.path.join(WORK_DIR, "package"))
print("ZIP size:", round(os.path.getsize(zip_path) / 1e6, 1), "MB")
files.download(zip_path)